# 🔍 AI Gateway 모니터링 종합 테스트

## 목적
APIM AI Gateway에서 다음을 App Insights에서 직접 확인합니다:

1. **토큰/TPM 모니터링** — 모델별·백엔드별 토큰 사용량 및 분당 토큰(TPM)
2. **프론트엔드/백엔드 요청·응답 로깅** — 실제 request/response body 포함

## 사전 조건
- APIM 배포 완료 (Lab 1~4)
- Application Insights 연동 완료
- `azure-openai-emit-token-metric` 정책 적용 완료
- APIM Diagnostics에 frontend/backend body logging 활성화

## 구조
| Phase | 내용 |
|-------|------|
| Phase 1 | 테스트 트래픽 생성 (여러 모델/백엔드로 분산) |
| Phase 2 | App Insights에서 토큰 메트릭 조회 (KQL) |
| Phase 3 | 모델별·백엔드별 TPM 집계 |
| Phase 4 | 프론트엔드/백엔드 요청·응답 로그 확인 |

In [ ]:
# ─── 환경 설정 ───
import os
import requests
import time
import json
from datetime import datetime, timedelta, timezone

# ✏️ 환경 설정 — source .env 또는 아래 값을 직접 입력하세요
APIM_URL = os.getenv("APIM_URL", "https://<apim-name>.azure-api.net")
SUBSCRIPTION_KEY = os.getenv("APIM_SUBSCRIPTION_KEY", "<your-subscription-key>")
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME", "gpt-4.1-nano")
API_VERSION = "2025-04-01-preview"

# App Insights 쿼리용 (Azure Portal → App Insights → API Access)
APP_INSIGHTS_APP_ID = os.getenv("APP_INSIGHTS_APP_ID", "<your-app-insights-app-id>")
APP_INSIGHTS_API_KEY = os.getenv("APP_INSIGHTS_API_KEY", "<your-app-insights-api-key>")

BASE_URL = f"{APIM_URL}/openai/deployments/{DEPLOYMENT_NAME}/chat/completions"
HEADERS = {
    "Content-Type": "application/json",
    "Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY
}

print("✅ 환경 설정 완료")
print(f"   APIM: {APIM_URL}")
print(f"   Model: {DEPLOYMENT_NAME}")
print(f"   API Version: {API_VERSION}")

---
## Phase 1: 테스트 트래픽 생성

여러 번 호출해서 3개 백엔드(East US, Sweden, West US)로 분산되는 트래픽을 만듭니다.  
각 호출마다 사용된 토큰 수와 백엔드를 기록합니다.

In [ ]:
def call_gateway(prompt, max_tokens=50):
    """AI Gateway 호출 → (status, tokens_info, backend, elapsed_ms) 반환"""
    start = time.time()
    try:
        resp = requests.post(
            BASE_URL,
            params={"api-version": API_VERSION},
            headers=HEADERS,
            json={
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": max_tokens
            },
            timeout=30
        )
        elapsed = int((time.time() - start) * 1000)
        backend = resp.headers.get("x-backend-url", "unknown")
        remaining = resp.headers.get("x-ratelimit-remaining-tokens", "N/A")
        
        tokens_info = {}
        if resp.status_code == 200:
            body = resp.json()
            usage = body.get("usage", {})
            tokens_info = {
                "prompt_tokens": usage.get("prompt_tokens", 0),
                "completion_tokens": usage.get("completion_tokens", 0),
                "total_tokens": usage.get("total_tokens", 0),
                "remaining_tokens": remaining
            }
        
        return resp.status_code, tokens_info, backend, elapsed
    except Exception as e:
        elapsed = int((time.time() - start) * 1000)
        return 0, {}, str(e), elapsed

print("✅ call_gateway 함수 준비 완료")

In [ ]:
# ─── 테스트 트래픽 생성 (15회 호출) ───
prompts = [
    ("What is Azure API Management?", 80),
    ("Explain load balancing in 2 sentences.", 60),
    ("서울의 인구는?", 30),
    ("What are tokens in LLM?", 100),
    ("List 3 benefits of AI Gateway.", 80),
    ("Describe circuit breaker pattern briefly.", 60),
    ("클라우드 컴퓨팅이란?", 50),
    ("How does semantic caching work?", 80),
    ("What is TPM in Azure OpenAI?", 60),
    ("Explain rate limiting.", 50),
    ("What is managed identity?", 40),
    ("Describe backend pool concept.", 70),
    ("Azure Monitor란 무엇인가요?", 60),
    ("Explain P50 P95 P99 latency.", 80),
    ("What is Application Insights?", 50),
]

results = []
total_tokens_all = 0
start_time = datetime.now(timezone.utc)

print(f"▶ 트래픽 생성 시작: {start_time.strftime('%H:%M:%S')} UTC")
print(f"  {len(prompts)}회 호출 예정\n")
print(f"{'#':>3}  {'Status':>6}  {'Prompt':>8}  {'Compl':>6}  {'Total':>6}  {'ms':>6}  Backend")
print("─" * 80)

for i, (prompt, max_tok) in enumerate(prompts, 1):
    code, tokens, backend, ms = call_gateway(prompt, max_tok)
    short_backend = backend.replace("https://", "").replace(".openai.azure.com", "")
    
    results.append({
        "call": i,
        "status": code,
        "prompt_tokens": tokens.get("prompt_tokens", 0),
        "completion_tokens": tokens.get("completion_tokens", 0),
        "total_tokens": tokens.get("total_tokens", 0),
        "remaining": tokens.get("remaining_tokens", "N/A"),
        "backend": short_backend,
        "latency_ms": ms
    })
    
    total_tokens_all += tokens.get("total_tokens", 0)
    icon = "✅" if code == 200 else f"❌{code}"
    print(f"{i:3d}  {icon:>6}  {tokens.get('prompt_tokens',0):>8}  {tokens.get('completion_tokens',0):>6}  {tokens.get('total_tokens',0):>6}  {ms:>5,}  {short_backend}")
    
    time.sleep(1)  # Rate limit 회피

end_time = datetime.now(timezone.utc)
elapsed_min = (end_time - start_time).total_seconds() / 60

print(f"\n═══════════════════════════════════════════")
print(f"  완료: {len([r for r in results if r['status']==200])}/{len(prompts)} 성공")
print(f"  총 토큰: {total_tokens_all:,}")
print(f"  소요 시간: {elapsed_min:.1f}분")
print(f"  실측 TPM: {int(total_tokens_all / max(elapsed_min, 0.01)):,} tokens/min")
print(f"═══════════════════════════════════════════")

In [ ]:
# ─── 백엔드별·모델별 토큰 집계 (로컬 데이터) ───
from collections import defaultdict

backend_stats = defaultdict(lambda: {"calls": 0, "prompt": 0, "completion": 0, "total": 0})

for r in results:
    if r["status"] == 200:
        b = r["backend"]
        backend_stats[b]["calls"] += 1
        backend_stats[b]["prompt"] += r["prompt_tokens"]
        backend_stats[b]["completion"] += r["completion_tokens"]
        backend_stats[b]["total"] += r["total_tokens"]

print("═" * 65)
print(" 백엔드별 토큰 사용량 (로컬 집계)")
print("═" * 65)
print(f"{'Backend':<30} {'Calls':>5} {'Prompt':>8} {'Compl':>8} {'Total':>8}")
print("─" * 65)

for backend, stats in sorted(backend_stats.items()):
    print(f"{backend:<30} {stats['calls']:>5} {stats['prompt']:>8} {stats['completion']:>8} {stats['total']:>8}")

print("─" * 65)
total_calls = sum(s['calls'] for s in backend_stats.values())
total_prompt = sum(s['prompt'] for s in backend_stats.values())
total_compl = sum(s['completion'] for s in backend_stats.values())
total_tok = sum(s['total'] for s in backend_stats.values())
print(f"{'합계':<30} {total_calls:>5} {total_prompt:>8} {total_compl:>8} {total_tok:>8}")

print(f"\n💡 이 데이터가 App Insights customMetrics에도 동일하게 기록되어야 합니다.")
print(f"   → 아래 Phase 2에서 KQL로 확인합니다.")

---
## Phase 2: App Insights에서 토큰 메트릭 조회

`azure-openai-emit-token-metric` 정책이 보내는 메트릭은 App Insights의 **customMetrics** 테이블에 저장됩니다.

### 방법 1: Azure Portal에서 직접 KQL 실행

Azure Portal → Application Insights → **Logs** 블레이드에서 아래 KQL을 실행하세요.

### 방법 2: App Insights REST API로 프로그래밍 방식 조회 (아래 셀)

### 📋 KQL 쿼리 모음 — Portal에서 바로 실행 가능

아래 KQL을 Azure Portal → App Insights → Logs에 붙여넣으세요.

#### 1) 모델별 토큰 사용량 (최근 1시간)
```kql
customMetrics
| where timestamp > ago(1h)
| where name startswith "ai-gateway-metrics"
| extend model = tostring(customDimensions["Model"])
| extend backend = tostring(customDimensions["Backend"])
| summarize 
    totalTokens = sum(value),
    promptTokens = sumif(value, name contains "Prompt"),
    completionTokens = sumif(value, name contains "Completion"),
    callCount = count()
    by model, backend
| order by totalTokens desc
```

#### 2) 모델별·백엔드별 TPM (분당 토큰) — 시계열
```kql
customMetrics
| where timestamp > ago(1h)
| where name startswith "ai-gateway-metrics"
| extend model = tostring(customDimensions["Model"])
| extend backend = tostring(customDimensions["Backend"])
| summarize TPM = sum(value) by model, backend, bin(timestamp, 1m)
| order by timestamp desc
| render timechart
```

#### 3) 백엔드별 Prompt vs Completion 토큰 분리
```kql
customMetrics
| where timestamp > ago(1h)
| where name startswith "ai-gateway-metrics"
| extend model = tostring(customDimensions["Model"])
| extend backend = tostring(customDimensions["Backend"])
| extend tokenType = case(
    name contains "Prompt", "Prompt",
    name contains "Completion", "Completion",
    "Total"
  )
| summarize tokens = sum(value) by backend, model, tokenType, bin(timestamp, 1m)
| render timechart
```

#### 4) 구독별 TPM 할당량 대비 사용률
```kql
let tpmLimit = 10000; // ai-gateway-policy.xml의 tokens-per-minute 설정값
customMetrics
| where timestamp > ago(1h)
| where name startswith "ai-gateway-metrics"
| extend subscriptionId = tostring(customDimensions["Subscription ID"])
| extend model = tostring(customDimensions["Model"])
| summarize TPM = sum(value) by subscriptionId, model, bin(timestamp, 1m)
| extend usagePercent = round(TPM * 100.0 / tpmLimit, 1)
| order by timestamp desc
```

#### 5) 429 Rate Limit 발생과 토큰 사용량 상관관계
```kql
let tokenUsage = customMetrics
| where timestamp > ago(1h)
| where name startswith "ai-gateway-metrics"
| summarize TPM = sum(value) by bin(timestamp, 1m);
let rateLimits = requests
| where timestamp > ago(1h)
| where resultCode == "429"
| summarize throttleCount = count() by bin(timestamp, 1m);
tokenUsage
| join kind=leftouter rateLimits on timestamp
| extend throttleCount = coalesce(throttleCount, 0)
| render timechart
```

In [ ]:
# ─── App Insights REST API로 KQL 실행 ───
# (Portal에서도 직접 실행 가능하지만, 노트북에서 자동화하려면 이 방법 사용)

def query_app_insights(kql: str) -> dict:
    """App Insights REST API를 통해 KQL 실행"""
    url = f"https://api.applicationinsights.io/v1/apps/{APP_INSIGHTS_APP_ID}/query"
    headers = {
        "x-api-key": APP_INSIGHTS_API_KEY,
        "Content-Type": "application/json"
    }
    resp = requests.post(url, headers=headers, json={"query": kql}, timeout=30)
    resp.raise_for_status()
    return resp.json()

def print_query_result(result: dict):
    """App Insights 쿼리 결과를 테이블로 출력"""
    tables = result.get("tables", [])
    if not tables:
        print("결과 없음")
        return
    
    table = tables[0]
    columns = [c["name"] for c in table["columns"]]
    rows = table["rows"]
    
    if not rows:
        print("결과 행이 없습니다 (데이터가 아직 수집되지 않았을 수 있음)")
        return
    
    # 간단한 테이블 출력
    col_widths = [max(len(str(c)), max(len(str(r[i])) for r in rows)) for i, c in enumerate(columns)]
    header = "  ".join(f"{c:<{w}}" for c, w in zip(columns, col_widths))
    print(header)
    print("─" * len(header))
    for row in rows:
        print("  ".join(f"{str(v):<{w}}" for v, w in zip(row, col_widths)))

print("✅ App Insights 쿼리 함수 준비 완료")
print(f"\n💡 API 키 설정이 안 되어 있으면 Portal에서 직접 KQL을 실행하세요.")

In [ ]:
# ─── 쿼리 1: 모델별·백엔드별 토큰 사용량 ───
print("═" * 60)
print(" 모델별·백엔드별 토큰 사용량 (App Insights customMetrics)")
print("═" * 60)

kql_tokens = """
customMetrics
| where timestamp > ago(1h)
| where name startswith "ai-gateway-metrics"
| extend model = tostring(customDimensions["Model"])
| extend backend = tostring(customDimensions["Backend"])
| summarize 
    totalTokens = sum(value),
    callCount = count()
    by model, backend
| order by totalTokens desc
"""

try:
    result = query_app_insights(kql_tokens)
    print_query_result(result)
except Exception as e:
    print(f"⚠️ API 조회 실패: {e}")
    print(f"\n📋 대신 Azure Portal → App Insights → Logs에서 위 KQL을 직접 실행하세요.")

In [ ]:
# ─── 쿼리 2: 모델별·백엔드별 TPM (분당 토큰) ───
print("═" * 60)
print(" 모델별·백엔드별 TPM — 분당 토큰 (최근 1시간)")
print("═" * 60)

kql_tpm = """
customMetrics
| where timestamp > ago(1h)
| where name startswith "ai-gateway-metrics"
| extend model = tostring(customDimensions["Model"])
| extend backend = tostring(customDimensions["Backend"])
| summarize TPM = sum(value) by model, backend, bin(timestamp, 1m)
| order by timestamp desc
"""

try:
    result = query_app_insights(kql_tpm)
    print_query_result(result)
except Exception as e:
    print(f"⚠️ API 조회 실패: {e}")
    print(f"\n📋 Azure Portal에서 직접 실행하세요.")

---
## Phase 3: 프론트엔드/백엔드 요청·응답 모니터링

APIM Diagnostics 설정에 `frontend`/`backend` 본문 로깅을 활성화했으므로,  
App Insights의 **requests** 및 **dependencies** 테이블에서 실제 요청/응답 내용을 확인할 수 있습니다.

### 데이터가 나타나는 위치

| App Insights 테이블 | 내용 |
|----------------------|------|
| `requests` | 프론트엔드 (클라이언트 → APIM) 요청/응답 |
| `dependencies` | 백엔드 (APIM → Azure OpenAI) 요청/응답 |
| `customMetrics` | `azure-openai-emit-token-metric` 토큰 메트릭 |
| `traces` | 정책 내 `trace` 로그 |

### 📋 프론트엔드/백엔드 KQL 쿼리 모음

#### 6) 프론트엔드 요청 목록 (클라이언트 → APIM)
```kql
requests
| where timestamp > ago(1h)
| project 
    timestamp,
    name,
    resultCode,
    duration,
    client_IP,
    requestBody = tostring(customDimensions["Request-Body"]),
    responseBody = tostring(customDimensions["Response-Body"]),
    subscriptionId = tostring(customDimensions["Subscription ID"])
| order by timestamp desc
```

#### 7) 백엔드 호출 목록 (APIM → Azure OpenAI)
```kql
dependencies
| where timestamp > ago(1h)
| project
    timestamp,
    target,
    name,
    resultCode,
    duration,
    requestBody = tostring(customDimensions["Request-Body"]),
    responseBody = tostring(customDimensions["Response-Body"])
| order by timestamp desc
```

#### 8) End-to-End 트랜잭션 추적 (프론트엔드 → 백엔드 조인)
```kql
requests
| where timestamp > ago(1h)
| project 
    timestamp,
    operation_Id,
    frontend_url = url,
    frontend_status = resultCode,
    frontend_duration = duration,
    client_IP
| join kind=inner (
    dependencies
    | where timestamp > ago(1h)
    | project
        operation_Id,
        backend_target = target,
        backend_status = resultCode,
        backend_duration = duration
) on operation_Id
| project-away operation_Id1
| extend overhead_ms = frontend_duration - backend_duration
| order by timestamp desc
```

#### 9) 백엔드별 성능 + 에러율 대시보드
```kql
dependencies
| where timestamp > ago(1h)
| extend backend = tostring(target)
| summarize
    totalCalls = count(),
    successCalls = countif(toint(resultCode) >= 200 and toint(resultCode) < 300),
    throttledCalls = countif(resultCode == "429"),
    errorCalls = countif(toint(resultCode) >= 500),
    avgLatency = avg(duration),
    p95Latency = percentile(duration, 95)
    by backend
| extend 
    successRate = round(successCalls * 100.0 / totalCalls, 1),
    throttleRate = round(throttledCalls * 100.0 / totalCalls, 1)
| order by totalCalls desc
```

#### 10) 모델별 AOAI TPM 현황 종합 (가장 중요!)
```kql
// 각 AOAI 인스턴스별로 실제 소비된 TPM을 정확히 추적
customMetrics
| where timestamp > ago(1h)
| where name startswith "ai-gateway-metrics"
| extend model = tostring(customDimensions["Model"])
| extend backend = tostring(customDimensions["Backend"])
| summarize TPM = sum(value) by model, backend, bin(timestamp, 1m)
| evaluate pivot(backend, sum(TPM))
| order by timestamp desc
```

In [ ]:
# ─── 쿼리 6: 프론트엔드 요청 목록 ───
print("═" * 60)
print(" 프론트엔드 요청 (클라이언트 → APIM) — 최근 10건")
print("═" * 60)

kql_frontend = """
requests
| where timestamp > ago(1h)
| project 
    timestamp,
    name,
    resultCode,
    duration = round(duration, 0),
    client_IP
| order by timestamp desc
| take 10
"""

try:
    result = query_app_insights(kql_frontend)
    print_query_result(result)
except Exception as e:
    print(f"⚠️ API 조회 실패: {e}")
    print(f"📋 Azure Portal → App Insights → Logs에서 직접 실행하세요.")

In [ ]:
# ─── 쿼리 7: 백엔드 호출 목록 ───
print("═" * 60)
print(" 백엔드 호출 (APIM → Azure OpenAI) — 최근 10건")
print("═" * 60)

kql_backend = """
dependencies
| where timestamp > ago(1h)
| project
    timestamp,
    target,
    resultCode,
    duration = round(duration, 0)
| order by timestamp desc
| take 10
"""

try:
    result = query_app_insights(kql_backend)
    print_query_result(result)
except Exception as e:
    print(f"⚠️ API 조회 실패: {e}")
    print(f"📋 Azure Portal → App Insights → Logs에서 직접 실행하세요.")

In [ ]:
# ─── 쿼리 9: 백엔드별 성능 + 에러율 ───
print("═" * 60)
print(" 백엔드별 성능 & 에러율 종합")
print("═" * 60)

kql_backend_perf = """
dependencies
| where timestamp > ago(1h)
| extend backend = tostring(target)
| summarize
    totalCalls = count(),
    successCalls = countif(toint(resultCode) >= 200 and toint(resultCode) < 300),
    throttledCalls = countif(resultCode == "429"),
    avgLatency = round(avg(duration), 0),
    p95Latency = round(percentile(duration, 95), 0)
    by backend
| extend 
    successRate = round(successCalls * 100.0 / totalCalls, 1),
    throttleRate = round(throttledCalls * 100.0 / totalCalls, 1)
| order by totalCalls desc
"""

try:
    result = query_app_insights(kql_backend_perf)
    print_query_result(result)
except Exception as e:
    print(f"⚠️ API 조회 실패: {e}")
    print(f"📋 Azure Portal → App Insights → Logs에서 직접 실행하세요.")

---
## Phase 4: Azure Portal에서 확인하는 방법 (수동 가이드)

### 🔹 방법 A: App Insights → Logs (KQL)

1. Azure Portal → **Application Insights** → 해당 리소스 선택
2. 왼쪽 메뉴 → **Logs** 클릭
3. 위 KQL 쿼리를 붙여넣고 **Run** 실행

### 🔹 방법 B: App Insights → Metrics (실시간 차트)

1. Azure Portal → **Application Insights** → 해당 리소스 선택
2. 왼쪽 메뉴 → **Metrics** 클릭
3. Metric Namespace: **azure.applicationinsights** 또는 **ai-gateway-metrics**
4. Metric: **Total Tokens**, **Prompt Tokens**, **Completion Tokens** 선택
5. **Apply splitting** → `Model`, `Backend` 차원으로 분리
6. 시간 범위를 **Last 1 hour**로 설정

### 🔹 방법 C: APIM → Analytics (내장 대시보드)

1. Azure Portal → **API Management** → 해당 인스턴스
2. 왼쪽 메뉴 → **Monitoring** → **Application Insights**
3. 또는 **Analytics** 블레이드에서 요청/응답 추이 확인

### 🔹 방법 D: APIM → Built-in Analytics

1. Azure Portal → **API Management** → 해당 인스턴스
2. 왼쪽 메뉴 → **Monitoring** → **Metrics**
3. Metric: **Requests**, **Duration**, **Capacity** 등
4. 필터: 특정 API, Backend, Status Code 등으로 필터

### 🔹 방법 E: End-to-End 트랜잭션 추적

1. Azure Portal → **Application Insights** → **Transaction search**
2. 특정 요청 클릭 → **End-to-end transaction details** 열기
3. 프론트엔드(request) → 백엔드(dependency) → 응답 전체 흐름 확인
4. 각 단계에서 **Request Body**, **Response Body** 확인 가능

---

## 📊 모니터링 아키텍처 요약

```
Client → APIM (Frontend)                       → Azure OpenAI (Backend)
         ├─ requests 테이블                      ├─ dependencies 테이블
         │  ├─ Request URL                       │  ├─ Backend Target (host)
         │  ├─ Request Body (프롬프트 포함)       │  ├─ Request Body
         │  ├─ Response Body                     │  ├─ Response Body (토큰 사용량 포함)
         │  ├─ Status Code                      │  ├─ Status Code
         │  ├─ Duration                          │  ├─ Duration
         │  └─ Client IP                         │  └─ operation_Id (조인 키)
         │                                       │
         └─ customMetrics 테이블 ─────────────────┘
            ├─ Total Tokens   (dimension: Model, Backend)
            ├─ Prompt Tokens  (dimension: Model, Backend)
            └─ Completion Tokens (dimension: Model, Backend)
                → bin(timestamp, 1m) = TPM (분당 토큰)
```

---
## ✅ 체크리스트

| # | 확인 항목 | KQL 테이블 | 확인 방법 |
|---|-----------|-----------|----------|
| 1 | 토큰 사용량 (Prompt/Completion/Total) | `customMetrics` | 쿼리 1 |
| 2 | 모델별·백엔드별 TPM | `customMetrics` | 쿼리 2, 10 |
| 3 | TPM 할당량 대비 사용률 | `customMetrics` | 쿼리 4 |
| 4 | 429 Rate Limit과 TPM 상관관계 | `customMetrics` + `requests` | 쿼리 5 |
| 5 | 프론트엔드 요청/응답 본문 | `requests` | 쿼리 6 |
| 6 | 백엔드 요청/응답 본문 | `dependencies` | 쿼리 7 |
| 7 | End-to-End 트랜잭션 (프론트→백엔드) | `requests` JOIN `dependencies` | 쿼리 8 |
| 8 | 백엔드별 성능/에러율 | `dependencies` | 쿼리 9 |
| 9 | AOAI 인스턴스별 TPM 피벗 테이블 | `customMetrics` | 쿼리 10 |